In [ ]:
#clear cache from memory and cuda
import torch
torch.cuda.empty_cache()

In [ ]:
import ipywidgets as widgets

import os
import csv
import shutil
import soundfile as sf
import numpy as np

from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_dataset
from evaluate import load as metrics_loader
from transformers import Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer
from transformers import WhisperForConditionalGeneration
from transformers import WhisperProcessor

import torch

W0403 22:24:58.367000 17872 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [ ]:
import re

def normalize_cs(text):
    text = text.lower()

    # keep English letters + Twi chars
    text = re.sub(r"[^a-z0-9ɔɛ\s']", "", text)

    # normalize apostrophes (optional)
    text = re.sub(r"'", "", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
wer_metric = metrics_loader("wer")

def get_wer(references, predictions, normalize=True, verbose=True):
  rs = references
  ps = predictions
  if normalize:
    ps = [normalize_cs(x) for x in predictions]
    rs = [normalize_cs(x) for x in references]
  if verbose:
    for r, p in zip(rs, ps):
      print(r)
      print(p)
      print()

  return wer_metric.compute(references=rs, predictions=ps)

In [ ]:
def count_trainable_parameters(model):
    model_parameters = filter(lambda p: p.requires_grad, model.parameters())
    params = sum([np.prod(p.size()) for p in model_parameters])
    return params

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "Kennethdot/ghana-english-twi-codeswitch-asr"
)

In [ ]:
from datasets import Audio

my_audio_dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

print(my_audio_dataset)
print(my_audio_dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['audio', 'speaker_id', 'prompt_set', 'age_range', 'gender', 'transcript', 'duration_seconds'],
        num_rows: 8590
    })
    validation: Dataset({
        features: ['audio', 'speaker_id', 'prompt_set', 'age_range', 'gender', 'transcript', 'duration_seconds'],
        num_rows: 6332
    })
    test: Dataset({
        features: ['audio', 'speaker_id', 'prompt_set', 'age_range', 'gender', 'transcript', 'duration_seconds'],
        num_rows: 6210
    })
})
{'audio': {'path': 'd051_std_1_20260203T130325Z.ogg', 'array': array([-1.82454678e-05, -5.21586626e-05, -8.06410098e-05, ...,
        1.49851409e-03,  2.71572149e-03,  1.67350820e-03]), 'sampling_rate': 16000}, 'speaker_id': 'd051', 'prompt_set': 'standard', 'age_range': 'nan', 'gender': 'nan', 'transcript': 'Do you have nsuo a ɛyɛ nwunu?', 'duration_seconds': 2.753499984741211}


In [ ]:

my_audio_dataset["validation"] = my_audio_dataset["validation"].shuffle(seed=42).select(range(225))
my_audio_dataset["test"] = my_audio_dataset["test"].shuffle(seed=42).select(range(225))

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

model = WhisperForConditionalGeneration.from_pretrained("GiftMark/akan-whisper-model")
processor = WhisperProcessor.from_pretrained("GiftMark/akan-whisper-model")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
LANGUAGE = None
TASK = 'transcribe'

print(f"Set LANGUAGE to: {LANGUAGE}")
print(f"Set TASK to: {TASK}")

Set LANGUAGE to: None
Set TASK to: transcribe


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('device is: ', device)

# for more efficient dataset processing
torch.set_num_threads(1)
torch.get_num_threads()
num_proc = os.cpu_count()
print('# processors:', num_proc)


device is:  cuda
# processors: 32


In [ ]:
%%time
def prepare_dataset(batch, processor=processor):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = processor.tokenizer(batch["transcript"]).input_ids
    batch["input_length"] = len(audio["array"]) / audio["sampling_rate"]
    return batch


my_audio_dataset = my_audio_dataset.map(prepare_dataset,
                                        writer_batch_size=64,
                                        num_proc=2,
                                        )

Map (num_proc=2):   0%|          | 0/225 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/225 [00:00<?, ? examples/s]

CPU times: total: 11.3 s
Wall time: 1min 43s


In [ ]:
print(my_audio_dataset)

DatasetDict({
    train: Dataset({
        features: ['audio', 'speaker_id', 'prompt_set', 'age_range', 'gender', 'transcript', 'duration_seconds', 'input_features', 'labels', 'input_length'],
        num_rows: 8590
    })
    validation: Dataset({
        features: ['audio', 'speaker_id', 'prompt_set', 'age_range', 'gender', 'transcript', 'duration_seconds', 'input_features', 'labels', 'input_length'],
        num_rows: 225
    })
    test: Dataset({
        features: ['audio', 'speaker_id', 'prompt_set', 'age_range', 'gender', 'transcript', 'duration_seconds', 'input_features', 'labels', 'input_length'],
        num_rows: 225
    })
})


In [ ]:
#@title Training Hyper Parameters
OUTPUT_DIR = './whisper_tuning' #@param
LOG_DIR = os.path.join(OUTPUT_DIR, 'logs')

LEARNING_RATE = 1e-5 #@param
BATCH_SIZE = 8 #@param
MAX_EPOCHS = 3 #@param
WARMUP_STEPS = 10 #@param
# set this as short as possible for your data
MAX_GEN_LEN = 32 #@param
# if save steps is 0, only last and best model will be written
SAVE_STEPS = 5 #@param

# see
# https://huggingface.co/docs/transformers/v4.46.2/en/main_classes/trainer#transformers.TrainingArguments
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    logging_dir=OUTPUT_DIR + '/logs',
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True,
    num_train_epochs=MAX_EPOCHS,
    #
    lr_scheduler_type='constant_with_warmup',
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    #
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=MAX_GEN_LEN,
    eval_steps=5,
    metric_for_best_model="wer",
    greater_is_better=False,
    #
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    logging_steps=1,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    #
    push_to_hub=False,
    remove_unused_columns=False,
    save_total_limit=3,
    dataloader_num_workers=4,      # parallel data loading
    dataloader_pin_memory=True,
    #eval_on_start=True,
)

In [ ]:
UPDATE_ENCODER = True #@param{type: 'boolean'}
UPDATE_DECODER = False #@param{type: 'boolean'}
UPDATE_PROJ = True #@param{type: 'boolean'}
model.model.encoder.requires_grad_(UPDATE_ENCODER)
model.model.decoder.requires_grad_(UPDATE_DECODER)
model.proj_out.requires_grad_(UPDATE_PROJ)


print('encoder params to update/total:', count_trainable_parameters(model.model.encoder), model.model.encoder.num_parameters())
print('decoder parans to update/total:', count_trainable_parameters(model.model.decoder), model.model.decoder.num_parameters())

print('overall # trainable parameters:', count_trainable_parameters(model))
print('overall # model parameters:', model.model.num_parameters())

encoder params to update/total: 88154112 88154112
decoder parans to update/total: 39832320 153580800
overall # trainable parameters: 127986432
overall # model parameters: 241734912


In [ ]:
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

In [ ]:
#@title Define Trainer
import evaluate
metric = evaluate.load("wer")
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)



trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=my_audio_dataset["train"],
    eval_dataset=my_audio_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor,
)


d:\project_kasa\venv\Lib\site-packages\accelerate\accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
# start tensorboard
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}

ERROR: Failed to launch TensorBoard (exited with 1).

In [ ]:
model = model.to(device)
print(f"Model device: {next(model.parameters()).device}")
print(f"CUDA memory allocated: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")
print(f"CUDA memory reserved:  {torch.cuda.memory_reserved(0)/1024**3:.2f} GB")

Model device: cuda:0
CUDA memory allocated: 0.90 GB
CUDA memory reserved:  0.99 GB


In [ ]:
batch = next(iter(trainer.get_train_dataloader()))
print({k: v.shape for k, v in batch.items()})

In [ ]:
trainer.train()

In [ ]:
print('evaluating best model after fine-tuning, lanuage:', LANGUAGE)
# Let Whisper handle multilingual input

results = trainer.evaluate(my_audio_dataset["validation"].shuffle(seed=42).select(range(200)))
print(results)

In [ ]:
output_dir = 'finetuned_whisper_model_akan_02' #@param {type: 'string'}
!mkdir -p {output_dir}

print('Saving model in:', output_dir)

# save model and processor, so we can later load as pretrained
save_model_dir = os.path.join(output_dir, 'saved_model')
trainer.model.save_pretrained(save_model_dir, safe_serialization=False)

# save processor also
save_processor_dir = os.path.join(output_dir, 'saved_processor')
processor.save_pretrained(save_processor_dir, safe_serialization=False)

In [ ]:
def transcribe_from_dataset(dataset_sample, whisper_model, max_new_tokens=128):
  input_features = processor.feature_extractor(
    dataset_sample["array"],
    sampling_rate=dataset_sample["sampling_rate"],
    return_tensors="pt").input_features

  predicted_ids = whisper_model.generate(
      input_features, max_new_tokens=max_new_tokens,
      task=TASK, forced_decoder_ids=None)
  transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
  return transcription[0].strip()

In [ ]:
default_model = WhisperForConditionalGeneration.from_pretrained("GiftMark/akan-whisper-model")
default_processor = WhisperProcessor.from_pretrained("GiftMark/akan-whisper-model")

In [ ]:
#@title Get WER on default and tuned model for comparison

save_pretrained_model_dir = save_model_dir # Use the already defined save_model_dir

finetuned_model = WhisperForConditionalGeneration.from_pretrained(save_pretrained_model_dir, local_files_only=True)

num_test_samples = 20 
normalize_for_wer_calc = True 

num_test_samples = min(num_test_samples, len(my_audio_dataset['test']))
print('number of test examples to process:', num_test_samples)

predictions = []
finetuned_predictions = []
references  = []

for idx in range(num_test_samples):
  print('inference on example:', idx)
  sample = my_audio_dataset['test'][idx]["audio"]
  predictions.append(transcribe_from_dataset(sample, default_model))
  finetuned_predictions.append(transcribe_from_dataset(sample, finetuned_model))
  references.append(my_audio_dataset['test'][idx]['transcript'])

default_wer = get_wer(references=references, predictions=predictions, normalize=normalize_for_wer_calc, verbose=False)
finetuned_wer = get_wer(references=references, predictions=finetuned_predictions, normalize=normalize_for_wer_calc, verbose=False)

print(f'DEFAULT WER: {default_wer}')
print(f'FINETUNED WER: {finetuned_wer}')

In [ ]:
#@title Run inference on individual example of test set
for idx in range(60, 90):
  print('inference on example:', idx)
  sample = my_audio_dataset['test'][idx]["audio"]
  transcript = my_audio_dataset['test'][idx]["transcript"]
  # use default_model or finetuned_model
  model = finetuned_model
  pred = transcribe_from_dataset(sample, model, max_new_tokens=32)
  print('Ground truth: ', transcript)
  print('  Prediction: ', pred)